In [2]:
import tensorflow as tf
import os
import glob
import numpy as np
from model import build_1d_cnn_model
from common import INPUT_SHAPE,OUTPUT_DIM_NOTES, fast_gpu_map,parse_filtered_audio_record
cnn_model=build_1d_cnn_model(None,INPUT_SHAPE,37,False)
cnn_model.summary()
tf.keras.utils.plot_model(cnn_model,to_file='cnn_model.png',show_shapes=True)
cnn_model.load_weights('/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/checkpoints/guitarmidi_epoch111_valAcc0.9598_valPrec0.7875_valRecall0.5495.h5')#
#cnn_model.load_weights('guitarmidi.weights.h5')


input_filepaths = '/home/gerald/workspace/src/GuitarMidi-LV2/python/neuralnetmodelling/training_subset'#sorted(glob.glob(os.path.join(input_data_dir, '**', 'input', 'data.tfrecord'), recursive=True))
# train_dataset = tf.data.Dataset.from_tensor_slices((input_filepaths))
# train_dataset=train_dataset.shuffle(buffer_size=len(input_filepaths))
# train_dataset=train_dataset.take(100)
# train_dataset = train_dataset.map(tf_load_sample_from_files, num_parallel_calls=tf.data.AUTOTUNE)
def representative_data_gen():
    # Use TFRecordDataset to actually read the files
    # We only need a few samples to calibrate quantization
    raw_dataset = tf.data.TFRecordDataset(input_filepaths).map(parse_filtered_audio_record).take(100)
    
    # Map using your existing loading function
    calib_dataset = raw_dataset.map(lambda path: fast_gpu_map(path, training=False))
    
    for input_value, _ in calib_dataset.batch(1):
        # input_value is the (312, 256, 1) tensor
        yield [input_value]

converter = tf.lite.TFLiteConverter.from_keras_model(cnn_model)

converter.optimizations = [ tf.lite.Optimize.DEFAULT]
# converter.representative_dataset = representative_data_gen
# converter.target_spec.supported_ops = [tf.lite.OpsSet.TFLITE_BUILTINS_INT8]
# converter.target_spec.supported_types = [tf.int8] 

# converter.target_spec.supported_types = [tf.float16]
# converter.target_spec._experimental_supported_accumulation_type = tf.dtypes.float16
# Perform the conversion
tflite_model = converter.convert()


# converter=tf.lite.TFLiteConverter.from_keras_model(cnn_model)
# converter.optimizations = [tf.lite.Optimize.DEFAULT]
# tflite_model=converter.convert()

with open('guitarmidi.tflite','wb') as f:
    f.write(tflite_model)
print("TFLite model saved as guitarmidi.tflite")

Image height:  148
Before string split: (None, 148, 512), max_x=148.0


Model: "guitar_note_detector"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_spectrogram   │ (None, 148, 256,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ local_mean          │ (None, 148, 256,  │          0 │ input_spectrogra… │
│ (AveragePooling2D)  │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ local_contrast      │ (None, 148, 256,  │          0 │ input_spectrogra… │
│ (Subtract)          │ 1)                │            │ local_mean[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_to_2d       │ (None, 148, 256,  │          0 │ local_contrast[0… │
│ (Reshape)           │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_compress_conv… │ (None, 148, 64,   │        136 │ reshape_to_2d[0]… │
│ (Conv2D)            │ 8)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_compress_bn    │ (None, 148, 64,   │         32 │ freq_compress_co… │
│ (BatchNormalizatio… │ 8)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_compress_act   │ (None, 148, 64,   │          0 │ freq_compress_bn… │
│ (LeakyReLU)         │ 8)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ freq_compress_drop  │ (None, 148, 64,   │          0 │ freq_compress_ac… │
│ (SpatialDropout2D)  │ 8)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ reshape_to_1d       │ (None, 148, 512)  │          0 │ freq_compress_dr… │
│ (Reshape)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tfm_block1_ln1      │ (None, 148, 512)  │      1,024 │ reshape_to_1d[0]… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tfm_block1_mha      │ (None, 148, 512)  │    131,776 │ tfm_block1_ln1[0… │
│ (MultiHeadAttentio… │                   │            │ tfm_block1_ln1[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tfm_block1_attn_add │ (None, 148, 512)  │          0 │ reshape_to_1d[0]… │
│ (Add)               │                   │            │ tfm_block1_mha[0… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tfm_block1_ln2      │ (None, 148, 512)  │      1,024 │ tfm_block1_attn_… │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tfm_block1_ffn1     │ (None, 148, 128)  │     65,664 │ tfm_block1_ln2[0… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tfm_block1_ffn_drop │ (None, 148, 128)  │          0 │ tfm_block1_ffn1[… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tfm_block1_ffn2     │ (None, 148, 512)  │     66,048 │ tfm_block1_ffn_d… │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ tfm_block1_ffn_add  │ (None, 148, 512)  │          0 │ tfm_block1_attn_

 Total params: 707,277 (2.70 MB)

 Trainable params: 703,293 (2.68 MB)

 Non-trainable params: 3,984 (15.56 KB)

INFO:tensorflow:Assets written to: /tmp/tmpnhqz83y9/assets


INFO:tensorflow:Assets written to: /tmp/tmpnhqz83y9/assets


Saved artifact at '/tmp/tmpnhqz83y9'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 148, 256, 1), dtype=tf.float32, name='input_spectrogram')
Output Type:
  TensorSpec(shape=(None, 37), dtype=tf.float32, name=None)
Captures:
  135512944981584: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135514807826768: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135503961465168: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135503961467088: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135514807826960: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135503961465744: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135503961464976: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135503961466704: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135503961464592: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135503961463824: TensorSpec(shape=(), dtype=tf.resource, name=None)
  13550396

W0000 00:00:1775557354.991118 2007512 tf_tfl_flatbuffer_helpers.cc:364] Ignored output_format.
W0000 00:00:1775557354.991143 2007512 tf_tfl_flatbuffer_helpers.cc:367] Ignored drop_control_dependency.
I0000 00:00:1775557354.991459 2007512 reader.cc:83] Reading SavedModel from: /tmp/tmpnhqz83y9
I0000 00:00:1775557354.995498 2007512 reader.cc:52] Reading meta graph with tags { serve }
I0000 00:00:1775557354.995514 2007512 reader.cc:147] Reading SavedModel debug info (if present) from: /tmp/tmpnhqz83y9
I0000 00:00:1775557355.045428 2007512 mlir_graph_optimization_pass.cc:437] MLIR V1 optimization pass is not enabled
I0000 00:00:1775557355.061570 2007512 loader.cc:236] Restoring SavedModel bundle.
I0000 00:00:1775557355.549440 2007512 loader.cc:220] Running initialization op on SavedModel bundle at path: /tmp/tmpnhqz83y9
I0000 00:00:1775557355.673787 2007512 loader.cc:471] SavedModel load for tags { serve }; Status: success: OK. Took 682343 microseconds.
I0000 00:00:1775557355.793007 200751